# Sapiens2 Normal → CoreML for LiToStudio  *(recommended)*

Converts Meta's **Sapiens2 surface-normal estimator** (April 2026) into a self-contained
`SapiensNormal.mlpackage` for LiToStudio's photo-measured mesh refinement. All Python
stays in this cloud notebook — the Mac side is pure Swift/CoreML.

**Runs on [Lightning AI](https://lightning.ai) or Google Colab** — CPU is enough (this
is conversion, not training).
- **Lightning**: open a **fresh CPU Studio** (the toolchain cell pins torch 2.5 CPU —
  Studios are persistent, so don't run this in a Studio you use for other work).
  The free 4-core Studio handles 0.4b/0.8b; for 1b pick a ≥32 GB machine.
- **Colab**: plain CPU runtime for 0.4b/0.8b; High-RAM for 1b.

**Why v2 over the original Sapiens**: mean angular error drops from 13.62° (v1-1b) /
12.38° (v1-2b) to **7.12° (v2-1b)** — ~46% lower — at the same 768×1024 input. Same
RenderPeople-style training domain, so everything LiToStudio expects still holds.
If this notebook breaks (repo churn), `sapiens_normal_coreml_colab.ipynb` converts v1.

**Baked into the wrapper** (so Swift feeds raw pixels and reads unit vectors):
- input: RGB image, 768×1024 (W×H), values 0–255 (CoreML `ImageType`)
- preprocessing: ImageNet×255 — `(x − [123.675, 116.28, 103.53]) / [58.395, 57.12, 57.375]`
- output: `normals` tensor `(1, 3, 1024, 768)`, **unit-normalized**
- channel convention: irrelevant — LiToStudio's `NormalRefine` self-calibrates axes per image.

**License**: the [Sapiens2 License](https://github.com/facebookresearch/sapiens2/blob/main/LICENSE.md)
is royalty-free incl. derivatives (no blanket non-commercial clause like v1's CC-BY-NC),
with prohibited uses (surveillance, biometric identification). Redistribution of the
converted model must carry the license.


In [ ]:
# ── Pick the model size ────────────────────────────────────────────────────
# 0.4b ≈ 0.8 GB fp16 — already expected to beat every v1 size.
# 0.8b ≈ 1.6 GB fp16 — best quality/RAM balance for a 16 GB Mac (recommended).
# 1b   ≈ 2.9 GB fp16 (1.46B params) — 7.12° flagship; tight next to the 7.4 GB DiT.
MODEL_SIZE = "0.8b"   # "0.4b" | "0.8b" | "1b"


In [ ]:
# ── Toolchain + the sapiens2 repo ──────────────────────────────────────────
# Order matters: the sapiens2 editable install may pull a newer CUDA torch as a
# dependency, so the known-good CPU pin is applied LAST (force-reinstall wins).
# (On Linux, coremltools prints "Failed to load _MLModelProxy / libcoremlpython"
#  warnings — that's the macOS-only prediction runtime. Harmless for conversion.)
%pip install -q coremltools==8.3 huggingface_hub safetensors
![ -d sapiens2 ] || git clone -q --depth 1 https://github.com/facebookresearch/sapiens2.git
%pip install -q -e ./sapiens2
%pip install -q --force-reinstall torch==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cpu

# Verify the *installed* dist (not the possibly-stale imported module):
from importlib.metadata import version
v = version("torch")
print("installed: torch", v, "| coremltools", version("coremltools"))
assert v.startswith("2.5."), f"torch pin failed (got {v}) — rerun this cell"
print("✓ Toolchain pinned. Now: Kernel ▸ Restart Kernel (once), then run the NEXT cell onward.")


In [ ]:
# ── Checkpoint from Hugging Face + model from the repo config ──────────────
import torch
assert torch.__version__.startswith("2.5."), (
    f"kernel still has torch {torch.__version__} loaded — Kernel ▸ Restart, then run from the MODEL_SIZE cell"
)
MODEL_SIZE = globals().get("MODEL_SIZE", "0.8b")   # default if the size cell was skipped after restart
print("model size:", MODEL_SIZE)

from huggingface_hub import list_repo_files, hf_hub_download

repo = f"facebook/sapiens2-normal-{MODEL_SIZE}"
files = [f for f in list_repo_files(repo) if f.endswith(".safetensors")]
assert files, f"no .safetensors in {repo}: {list_repo_files(repo)}"
print("checkpoint:", files[0])
ckpt_path = hf_hub_download(repo, files[0])

from sapiens.dense.src.models.init_model import init_model
config = (f"sapiens2/sapiens/dense/configs/normal/metasim_render_people/"
          f"sapiens2_{MODEL_SIZE}_normal_metasim_render_people-1024x768.py")
model = init_model(config, ckpt_path, device="cpu")
model = model.eval()


In [ ]:
# ── Wrap: bake normalization + upsample + unit-normalize into the graph ────
import torch.nn as nn
import torch.nn.functional as F

class WrappedSapiens2Normal(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.register_buffer("mean", torch.tensor([123.675, 116.28, 103.53]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([58.395, 57.12, 57.375]).view(1, 3, 1, 1))

    def forward(self, x):                     # x: (1,3,1024,768) RGB in 0..255
        x = (x - self.mean) / self.std
        n = self.m(x)                          # NormalEstimator.forward → (1,3,H,W)
        n = F.interpolate(n, size=(1024, 768), mode="bilinear", align_corners=False)
        n = n / n.norm(dim=1, keepdim=True).clamp(min=1e-5)
        return n

wrapped = WrappedSapiens2Normal(model).eval()
example = torch.rand(1, 3, 1024, 768) * 255
with torch.no_grad():
    traced = torch.jit.trace(wrapped, example)
    out = traced(example)
print("traced ✓  output", tuple(out.shape), "| per-pixel L2 ≈", float(out.norm(dim=1).mean()))
assert tuple(out.shape) == (1, 3, 1024, 768)


In [ ]:
# ── Convert to CoreML (fp16 mlprogram) ─────────────────────────────────────
mlmodel = ct.convert(
    traced,
    inputs=[ct.ImageType(name="image", shape=(1, 3, 1024, 768),
                         scale=1.0, color_layout=ct.colorlayout.RGB)],
    outputs=[ct.TensorType(name="normals")],
    minimum_deployment_target=ct.target.macOS15,
    compute_precision=ct.precision.FLOAT16,
    convert_to="mlprogram",
)
mlmodel.user_defined_metadata["lito.normalized-input"] = "1"   # Swift: feed raw 0–255
mlmodel.user_defined_metadata["lito.model"] = f"sapiens2-normal-{MODEL_SIZE}"
mlmodel.short_description = (
    "Meta Sapiens2 surface-normal estimator wrapped for LiToStudio: "
    "RGB 0-255 in, unit normal map (1,3,1024,768) out. Sapiens2 License."
)
mlmodel.save("SapiensNormal.mlpackage")
!du -sh SapiensNormal.mlpackage


In [ ]:
# ── Package for download (works on Lightning AI and Colab) ─────────────────
!zip -r -q SapiensNormal.mlpackage.zip SapiensNormal.mlpackage
!ls -lh SapiensNormal.mlpackage.zip

import importlib.util, os
if importlib.util.find_spec("google.colab"):
    # Colab. For 0.8b/1b, Drive is far more reliable than the browser download:
    #   from google.colab import drive; drive.mount('/content/drive')
    #   !cp SapiensNormal.mlpackage.zip /content/drive/MyDrive/
    from google.colab import files
    files.download("SapiensNormal.mlpackage.zip")
else:
    print("Lightning AI: right-click the zip in the left file browser → Download, or:")
    print(f"  lightning download {os.path.abspath('SapiensNormal.mlpackage.zip')}")


## Install on the Mac

```bash
cd ~/Downloads/LiToStudio
unzip ~/Downloads/SapiensNormal.mlpackage.zip -d weights/
```

**Verify** (writes a normal-map visualization of an existing cutout):

```bash
swift build -c release
BIN=$(swift build -c release --show-bin-path); cp -f weights/mlx.metallib "$BIN/"
"$BIN/LiToSmoke" normals weights TMP/quality30.rmbg.png TMP/normals_vis.png
```

`TMP/normals_vis.png` should look like a classic normal map of the person (smooth
gradients over the body, crisp cloth folds). Then run the quality pipeline:

```bash
./run.sh sculpt testset/17.jpg out/17 25 3.0 7 3
```

and check the `[refine]` line: axis calibration should report `Σ|corr| ≥ ~1.2`.
If it aborts with a weak-correlation message, ping Claude with the log line.
